# Stage 3.5 — Dataset Integrity & Sampling Audit Analysis

**Project:** Persona Engine — Stage 3.5
**Objective:** Deeper diagnostic audit inspecting target repetition, context window multiplication, conversation dominance, cross-split consistency, and semantic prompt ambiguity.
**Source:** `ml/data/processed/dataset_integrity_audit.json`

In [ ]:
import sys
import json
from pathlib import Path

ml_root = Path.cwd().resolve()
if ml_root.name != "ml" and (ml_root / "ml").is_dir():
    ml_root = ml_root / "ml"
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

audit_path = ml_root / "data" / "processed" / "dataset_integrity_audit.json"
with open(audit_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Audit report loaded successfully.")
print("Unique target message IDs:", data["unique_target_message_analysis"]["total_unique_target_message_ids"])
print("Total candidate examples:", data["unique_target_message_analysis"]["total_candidate_examples"])

## 1. Target Message Context Multiplier Distribution

In [ ]:
occ = data["unique_target_message_analysis"]["target_occurrence_distribution"]
pct = data["unique_target_message_analysis"]["target_occurrence_percentages"]
print("=" * 50)
print("TARGET OCCURRENCE MULTIPLICATION")
print("=" * 50)
for k, cnt in occ.items():
    bar = "#" * int(cnt / 30)
    print(f"{k:5} | {cnt:5,} targets ({pct[k]:5.1f}%) | {bar}")

## 2. Conversation Dominance in Training Set

In [ ]:
dom = data["conversation_dominance_train"]
print("=" * 50)
print("CUMULATIVE CONVERSATION DOMINANCE (TRAIN)")
print("=" * 50)
for k, item in dom.items():
    bar = "#" * int(item["percentage_of_training_set"])
    print(f"{k:8} | {item['example_count']:5,} ex ({item['percentage_of_training_set']:5.1f}%) | {bar}")

## 3. Cross-Split Consistency (Context Depth & Language)

In [ ]:
ctx = data["context_depth_distribution_by_split"]
print("=" * 60)
print(f"{'Depth':10} | {'Train %':12} | {'Val %':12} | {'Test %':12}")
print("=" * 60)
for d in ["1_turn", "2_turn", "4_turn", "6_turn"]:
    tr_p = ctx["train"][d]["percentage"]
    va_p = ctx["validation"][d]["percentage"]
    te_p = ctx["test"][d]["percentage"]
    print(f"{d:10} | {tr_p:10.1f}% | {va_p:10.1f}% | {te_p:10.1f}%")

## 4. Repeated Short Response Concentration

In [ ]:
rep_conc = data["repeated_short_responses"]["split_concentration"]
print("=" * 50)
print("TOP 20 REPEATED TARGETS CONCENTRATION")
print("=" * 50)
for s_name in ["train", "validation", "test"]:
    info = rep_conc[s_name]
    print(f"{s_name.capitalize():12} | {info['top_20_count']:4,} / {info['total_count']:4,} ({info['percentage']:5.1f}%)")

## 5. Duplicate Semantic Contexts

In [ ]:
sem = data["duplicate_semantic_contexts"]
print("=" * 50)
print("TOP MULTI-TARGET USER PROMPTS")
print("=" * 50)
for item in sem["top_multi_target_prompts"][:8]:
    print(f"{repr(item['prompt_pattern']):25} -> {item['distinct_target_count']:2} distinct responses")